In [ ]:
import os
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from sec_edgar_downloader import Downloader
import time
import sqlite3

In [ ]:
# Define download directories
DOWNLOAD_DIR = "sec-edgar-filings"
FOLDER_1 = "C:/Users/sudet/Desktop/finai/finai/Data Collection/sec-edgar-filings"
FOLDER_2 = "C:/Users/sudet/Desktop/to-be-processed"

In [ ]:
# Load tickers from CSV
file_path = "companies.csv"  # Update this path if needed
companies_df = pd.read_csv(file_path, header=None)
companies_df = companies_df.iloc[1500:1750]
tickers = companies_df.iloc[1:, 1].tolist()  # Extract tickers (ignoring header)
cik_dict = {row[1]: str(row[0]).zfill(10) for _, row in companies_df.iloc[1:].iterrows()}  # Map Ticker -> Padded CIK

In [ ]:
def filing_exists(ticker, form, cik, year):
    """
    Check if the filing for a specific ticker, form, CIK, and year already exists 
    in either FOLDER_1, FOLDER_2, or the database.
    """
    paths_to_check = [
        os.path.join(FOLDER_1, ticker, form)
    ]
    
    cik_str = str(cik).zfill(10)  # Ensures leading zeros (e.g., 320193 -> 0000320193)
    year_suffix = str(year)[-2:]  # Convert year to 2-digit format (e.g., 2017 -> "17")

    print(f"\nChecking for {ticker} ({form}, {year}) - CIK: {cik_str}")  # Debug Print

    for base_path in paths_to_check:
        print(f"📂 Looking in: {base_path}")  # Debug print

        if not os.path.exists(base_path):
            print(f"Path does not exist: {base_path}")
            continue  # Skip to the next folder if this one doesn't exist

        found_folders = os.listdir(base_path)
        print(f"📌 Found {len(found_folders)} folders in {base_path}: {found_folders}")  # Debug print

        for folder in found_folders:
            #print(f"   📁 Checking folder: {folder}")  # Debug print
            
            # Expected format: "0000320193-17-000009"
            if folder.startswith(f"{cik_str}-{year_suffix}"):
                print(f"✅ Match found: {folder} for {ticker} in {base_path}")
                return True  # Found a matching filing

    print(f"No match found for {ticker} in {paths_to_check}")  # Debugging output
    return False  # No matching filing found

In [ ]:
years = range(2000, 2025)  # Adjust as needed
forms = ["10-Q"]  # Forms to download

In [ ]:
def download_filings(cik, ticker):
    email_address = "hitiras@hotmail.com"  # Replace with your email address
    dl = Downloader(DOWNLOAD_DIR, email_address)

    for form in forms:
        for year in years:
            if filing_exists(ticker, form, cik, year):
                print(f"Skipping {form} for {ticker} ({year}) - Already Downloaded")
                continue

            try:
                print(f"Downloading {form} for {ticker} ({year})...")
                time.sleep(1)  # Avoid SEC rate limits
                dl.get(form, cik, after=f"{year}-01-01", before=f"{year}-12-31")
                #time.sleep(5)  # Avoid SEC rate limits
                
            except Exception as e:
                print(f"Error downloading {form} for {ticker} in {year}: {e}")
                time.sleep(30)  # Sleep longer if an error occurs

In [ ]:
def main():
    max_threads = 1  # Reduce concurrency to prevent SEC rate limits
    with ThreadPoolExecutor(max_workers=max_threads) as executor:
        futures = {executor.submit(download_filings, cik_dict[ticker], ticker): ticker for ticker in tickers}

        for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading SEC Filings"):
            ticker = futures[future]
            try:
                future.result()  # Ensure exceptions are caught
            except Exception as e:
                print(f"Error downloading {ticker}: {e}")
                time.sleep(30)

    print("Download complete!")


In [ ]:
print("Strating SEC filings download...")
if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd 
import os
# Paths
DOWNLOAD_DIR = "C:/Users/sudet/Desktop/finai/finai/Data Collection/sec-edgar-filings"
file_path = "companies.csv"  # Update with the actual path to your CSV file

# Load CIK-to-Ticker mapping from CSV
companies_df = pd.read_csv(file_path, header=None)
cik_dict = {str(row[0]).zfill(10): row[1] for _, row in companies_df.iloc[1:].iterrows()}  # CIK -> Ticker

# Rename CIK folders to Ticker names
for cik_folder in os.listdir(DOWNLOAD_DIR):
    cik_path = os.path.join(DOWNLOAD_DIR, cik_folder)

    # Ensure it's a directory and a valid CIK
    if os.path.isdir(cik_path) and cik_folder in cik_dict:
        ticker_name = cik_dict[cik_folder]  # Get corresponding ticker
        new_path = os.path.join(DOWNLOAD_DIR, ticker_name)

        if not os.path.exists(new_path):  # Avoid overwriting existing folders
            os.rename(cik_path, new_path)
            print(f"✅ Renamed {cik_folder} ➝ {ticker_name}")
        else:
            print(f"⚠️ Skipped {cik_folder} (Ticker {ticker_name} already exists!)")
    else:
        print(f"❌ Skipped {cik_folder} (No matching ticker found)")


In [ ]:
import os
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from sec_edgar_downloader import Downloader
import time
import sqlite3

# Define download directories
DOWNLOAD_DIR = "sec-edgar-filings"
FOLDER_1 = "C:/Users/sudet/Desktop/finai/finai/Data Collection/sec-edgar-filings"
FOLDER_2 = "C:/Users/sudet/Desktop/to-be-processed"

# Load tickers from CSV
file_path = "companies.csv"
companies_df = pd.read_csv(file_path, header=None)
companies_df = companies_df.iloc[0:500]
tickers = companies_df.iloc[1:, 1].tolist()
cik_dict = {row[1]: str(row[0]).zfill(10) for _, row in companies_df.iloc[1:].iterrows()}

"""def filing_exists(ticker, form, cik, year):
    paths_to_check = [
        os.path.join(FOLDER_1, ticker, form)
    ]
    cik_str = str(cik).zfill(10)
    year_suffix = str(year)[-2:]

    for base_path in paths_to_check:
        if not os.path.exists(base_path):
            continue

        found_folders = os.listdir(base_path)
        for folder in found_folders:
            if folder.startswith(f"{cik_str}-{year_suffix}"):
                return True
    return False"""
    
def filing_exists(ticker, form, cik, year):
    """
    Simplified: Only checks if the top-level ticker folder exists in FOLDER_1.
    Ignores form, cik, and year — assumes that if ticker folder exists, filings were downloaded.
    """
    folder_path = os.path.join(FOLDER_1, ticker)
    return os.path.exists(folder_path)


years = range(2010, 2025)
forms = ["10-Q"]

def download_filings(cik, ticker):
    email_address = "hitiras@hotmail.com"
    dl = Downloader(DOWNLOAD_DIR, email_address)

    for form in forms:
        filings_downloaded = False
        for year in years:
            if filing_exists(ticker, form, cik, year):
                print(f"Skipping {form} for {ticker} - Already Downloaded")
                continue
            else:
                if not filings_downloaded:
                    try:
                        print(f"Downloading {form} for {ticker}... (all years)")
                        time.sleep(1)
                        dl.get(form, cik, after=f"{year}-01-01", before=f"{year}-12-31")
                        filings_downloaded = True
                    except Exception as e:
                        print(f"Error downloading {form} for {ticker}: {e}")
                        time.sleep(30)
                break

def main():
    max_threads = 5
    with ThreadPoolExecutor(max_workers=max_threads) as executor:
        futures = {executor.submit(download_filings, cik_dict[ticker], ticker): ticker for ticker in tickers}

        for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading SEC Filings"):
            ticker = futures[future]
            try:
                future.result()
            except Exception as e:
                print(f"Error downloading {ticker}: {e}")
                time.sleep(30)

    print("Download complete!")

print("Starting SEC filings download...")
if __name__ == "__main__":
    main()

In [ ]:
import os
import pandas as pd
import time
from tqdm import tqdm
from sec_edgar_downloader import Downloader
from concurrent.futures import ThreadPoolExecutor, as_completed

# Constants
DOWNLOAD_DIR = "sec-edgar-filings"
FORMS = ["10-Q"]
YEARS = range(2010, 2025)
MAX_THREADS = 10
EMAIL = "hitiras@hotmail.com"

# Load tickers and CIKs from CSV
file_path = "companies.csv"
companies_df = pd.read_csv(file_path, header=None)
companies_df = companies_df.iloc[500:1000]
tickers = companies_df.iloc[1:, 1].tolist()
cik_dict = {row[1]: str(row[0]).zfill(10) for _, row in companies_df.iloc[1:].iterrows()}

# Download all filings for all years
def download_filings(cik, ticker):
    dl = Downloader(DOWNLOAD_DIR, EMAIL)

    for form in FORMS:
        for year in YEARS:
            try:
                print(f"Downloading {form} for {ticker} - {year}")
                time.sleep(1)  # polite delay
                dl.get(form, cik, after=f"{year}-01-01", before=f"{year}-12-31")
            except Exception as e:
                print(f"Error downloading {form} for {ticker} in {year}: {e}")
                time.sleep(30)

# Main execution
def main():
    with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
        futures = {executor.submit(download_filings, cik_dict[ticker], ticker): ticker for ticker in tickers}

        for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading SEC Filings"):
            ticker = futures[future]
            try:
                future.result()
            except Exception as e:
                print(f"Error with {ticker}: {e}")
                time.sleep(30)

if __name__ == "__main__":
    print("Starting bulk 10-Q download for 2010–2024...")
    main()
